# NHA Hackathon – Problem Statement 01  
## Clinical Document Classification & Compliance to STG requirements

This notebook is a **starter scaffold** for participants working on:

- mixed-quality healthcare document ingestion
- OCR + layout understanding
- visual cue detection
- STG / policy rule checks
- explainable claim decisioning
- episode timeline construction
- extra / non-required document identification


### Deliverables this notebook is designed to help produce
1. **Per-page/package JSON output** in the exact format required by the problem statement  
2. **Human-readable summary table** with document type, rule checks, and reasons  
3. **Episode timeline** with admission / investigation / procedure / discharge ordering  
4. **Decision**: `PASS`, `CONDITIONAL`, or `FAIL` with evidence and confidence

This notebook is a skeleton and can be changed by participants 

In [ ]:
# =========================
# 1. INSTALLS / IMPORTS
# =========================

# Run once on the sandbox. Safe to re-run (pip is idempotent).
# !pip install pymupdf easyocr opencv-python pillow pandas numpy rapidfuzz python-dateutil

from __future__ import annotations

import os
import re
import io
import json
import math
import uuid
import base64
import hashlib
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import pandas as pd
import numpy as np

import fitz  # PyMuPDF
from PIL import Image
import cv2

# easyocr is heavy to import; we initialize the reader lazily in get_reader()
import easyocr

## Download the Dataset
We have provided a dedicated widget to download the hackathon datasets directly from the platform into this notebook environment.

### 1. Import the Widget

from databank_download_widget import DatabankDownloadWidget

### 2. Download the Databank
Select the cell below and run it.
Enter the Databank ID for the hackathon package.
Enter your email and password for the platform.
Click the **Download** button.

The widget will download and unzip the data right into your current directory. You can monitor the progress in the status output area below the button.

### **Databank ID for PS1: c110a5f8-6e79-43bd-bd7a-979677354958**

In [ ]:
from databank_download_widget import DatabankDownloadWidget

# Create an instance of the databank widget
databank_downloader = DatabankDownloadWidget()

# Display the widget
databank_downloader.display()

In [ ]:
# =========================
# 2. CONFIG
# =========================

DATA_ROOT = Path("./Claims")         # input claims folder
OUTPUT_ROOT = Path("./outputs")    # json/csv/html outputs
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

PACKAGE_CODES = ["MG064A", "SG039C", "MG006A", "SB039A"]

DECISION_PASS = "PASS"
DECISION_CONDITIONAL = "CONDITIONAL"
DECISION_FAIL = "FAIL"

# ----- VLM / OCR runtime -----
VLM_MODEL = "Gemma 3 12B"     # primary multimodal model
VLM_FALLBACK = "Gemma 3 4B"   # smaller fallback when primary fails / errors
VLM_CONCURRENCY = 4           # matches sandbox CPU budget (4 cores requested)
MAX_IMAGE_SIDE = 1280         # downscale large pages before sending to VLM
PDF_RENDER_ZOOM = 2.0         # 2x zoom ~ 144 DPI, good OCR/VLM tradeoff

VLM_CACHE_DIR = OUTPUT_ROOT / "vlm_cache"
VLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Per-case package code is filled in by discover_cases as a side effect so
# the cell-36 runner can resolve package_code via PACKAGE_CODE_LOOKUP[case_id].
PACKAGE_CODE_LOOKUP: Dict[str, str] = {}

# Lazy easyocr reader so notebook import is fast and the model is loaded once.
_READER = None
def get_reader():
    global _READER
    if _READER is None:
        _READER = easyocr.Reader(["en"], gpu=False, verbose=False)
    return _READER

### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere

In [ ]:
from nha_client import NHAclient

# Fill these in the sandbox before running. Local dev: leave blank and skip
# this cell; analyze_page_with_vlm will fall back to a deterministic skeleton.
clientId = ""
clientSecret = ""

nc = None
if clientId and clientSecret:
    nc = NHAclient(clientId, clientSecret)
    print("NHAclient ready.")
else:
    print("NHAclient not initialized (clientId/clientSecret empty).")

In [ ]:
# =========================
# 3. OUTPUT SCHEMAS
# =========================
# These are the exact expected keys per package, based on the provided examples.

PACKAGE_SCHEMAS = {
    "MG064A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "cbc_hb_report", "indoor_case",
        "treatment_details", "post_hb_report", "discharge_summary",
        "severe_anemia", "common_signs", "significant_signs",
        "life_threatening_signs", "extra_document", "document_rank"
    ],
    "SG039C": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "usg_report", "lft_report", "operative_notes",
        "pre_anesthesia", "discharge_summary", "photo_evidence",
        "histopathology", "clinical_condition", "usg_calculi",
        "pain_present", "previous_surgery", "extra_document", "document_rank"
    ],
    "MG006A": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "investigation_pre", "pre_date", "vitals_treatment",
        "investigation_post", "post_date", "discharge_summary", "poor_quality",
        "fever", "symptoms", "extra_document", "document_rank"
    ],
    "SB039A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes",
        "implant_invoice", "post_op_photo", "post_op_xray", "discharge_summary",
        "doa", "dod", "arthritis_type", "post_op_implant_present",
        "age_valid", "extra_document", "document_rank"
    ],
}

KEY_ALIASES = {
    "S3_link": "link",
    "s3_link": "link",
    "S3_link/DocumentName": "link",
}

# ---------------------------------------------------------------------------
# Static package metadata. Single source of truth read by:
#   - analyze_page_with_vlm (prompt construction)
#   - populate_row_for_package (binary key population + regex backstop)
#   - run_rules_engine (STG checks)
# ---------------------------------------------------------------------------

# Which schema key holds the document link, per package.
PACKAGE_LINK_KEY = {
    "MG064A": "link",
    "SG039C": "S3_link/DocumentName",
    "MG006A": "S3_link/DocumentName",
    "SB039A": "link",
}

# Per-package allowed doc-type labels. The VLM is constrained to choose one.
# These intersect DOCUMENT_TYPES (cell 15) with each schema's binary doc keys
# and always include "extra_document".
PACKAGE_DOC_TYPES = {
    "MG064A": [
        "clinical_notes", "cbc_hb_report", "indoor_case",
        "treatment_details", "post_hb_report", "discharge_summary",
        "extra_document",
    ],
    "SG039C": [
        "clinical_notes", "usg_report", "lft_report", "operative_notes",
        "pre_anesthesia", "discharge_summary", "photo_evidence",
        "histopathology", "extra_document",
    ],
    "MG006A": [
        "clinical_notes", "investigation_pre", "vitals_treatment",
        "investigation_post", "discharge_summary", "extra_document",
    ],
    "SB039A": [
        "clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes",
        "implant_invoice", "post_op_photo", "post_op_xray",
        "discharge_summary", "extra_document",
    ],
}

# Non-document binary keys per package (clinical findings, STG flags, etc.).
PACKAGE_EVIDENCE_KEYS = {
    "MG064A": ["severe_anemia", "common_signs", "significant_signs", "life_threatening_signs"],
    "SG039C": ["clinical_condition", "usg_calculi", "pain_present", "previous_surgery"],
    "MG006A": ["poor_quality", "fever", "symptoms"],
    "SB039A": ["arthritis_type", "post_op_implant_present", "age_valid"],
}

# Date keys per package (output schema).
PACKAGE_DATE_KEYS = {
    "MG064A": [],
    "SG039C": [],
    "MG006A": ["pre_date", "post_date"],
    "SB039A": ["doa", "dod"],
}

# Clinical synonym dictionary used as a regex backstop. If the VLM returns 0
# for a key but a synonym is found in the corrected text, we flip to 1.
EVIDENCE_SYNONYMS = {
    "MG064A": {
        "severe_anemia": ["severe anemia", "severe anaemia", "hb <", "hb<", "haemoglobin <", "hemoglobin <"],
        "common_signs": ["pallor", "fatigue", "weakness", "tiredness", "dizziness", "lethargy"],
        "significant_signs": ["tachycardia", "breathlessness", "dyspnea", "dyspnoea", "syncope", "palpitation"],
        "life_threatening_signs": ["cardiac failure", "shock", "severe hypoxia", "icu", "critical", "altered sensorium"],
    },
    "SG039C": {
        "clinical_condition": ["cholecyst", "gall bladder", "biliary", "cholelithiasis", "gallbladder"],
        "usg_calculi": ["calculi", "calculus", "stone", "gallstone", "gb stone", "gb calculi"],
        "pain_present": ["pain", "colic", "ruq", "right upper quadrant", "rt hypochondrium", "tenderness"],
        "previous_surgery": ["previous surgery", "h/o surgery", "past surgery", "operated", "k/c/o surgery"],
    },
    "MG006A": {
        "fever": ["fever", "pyrexia", "febrile", "temperature", "temp 1", "temp 10"],
        "symptoms": ["headache", "vomiting", "nausea", "abdominal pain", "diarrhea", "diarrhoea",
                     "chills", "rigors", "loose motion", "loss of appetite", "weakness", "myalgia"],
    },
    "SB039A": {
        "arthritis_type": ["osteoarthritis", "rheumatoid", "arthritis", "degenerative joint", "oa knee", "tricompartmental"],
        "post_op_implant_present": ["implant", "prosthesis", "tkr", "total knee replacement",
                                    "knee arthroplasty", "femoral component", "tibial component"],
        "age_valid": [],  # handled numerically via find_age + STG age range
    },
}

# STG age windows. Ages outside the range -> age_valid = 0.
# SB039A: lower bound 45 per NHA TKR STG; no defensible upper cap so we use 120
# (was 90 previously — that incorrectly invalidated valid >90 TKR cases).
PACKAGE_AGE_RANGE = {
    "MG064A": (0, 120),
    "SG039C": (0, 120),
    "MG006A": (0, 120),
    "SB039A": (45, 120),
}

# Doc-type gates for the regex synonym backstop in populate_row_for_package.
# A key listed here will only get its 0->1 backstop bump if the page's doc_type
# is in the allowed set. This prevents e.g. "calculi" mentioned in a clinical
# note from flipping usg_calculi when the page itself is not a USG report.
# Keys NOT in this dict are unrestricted (general clinical findings).
EVIDENCE_DOC_GATES = {
    # SG039C
    "usg_calculi": {"usg_report"},
    "previous_surgery": {"clinical_notes", "discharge_summary", "operative_notes"},
    # SB039A
    "arthritis_type": {"clinical_notes", "xray_ct_knee", "operative_notes", "discharge_summary"},
    "post_op_implant_present": {"operative_notes", "implant_invoice", "post_op_xray", "post_op_photo", "discharge_summary"},
}

In [ ]:
# =========================
# 4. DATA MODELS
# =========================

@dataclass
class OCRLine:
    text: str
    bbox: Optional[List[int]] = None
    confidence: Optional[float] = None

@dataclass
class PageResult:
    case_id: str
    file_name: str
    page_number: int
    extracted_text: str = ""
    ocr_lines: List[OCRLine] = field(default_factory=list)
    doc_type: str = "unknown"
    doc_type_confidence: float = 0.0
    visual_tags: Dict[str, Any] = field(default_factory=dict)
    entities: Dict[str, Any] = field(default_factory=dict)
    quality: Dict[str, Any] = field(default_factory=dict)
    output_row: Dict[str, Any] = field(default_factory=dict)
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TimelineEvent:
    sequence: int
    event_type: str
    date: Optional[str]
    source_document: str
    temporal_validity: str
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ClaimDecision:
    case_id: str
    package_code: str
    decision: str
    confidence: float
    reasons: List[str]
    missing_documents: List[str] = field(default_factory=list)
    rule_flags: List[str] = field(default_factory=list)
    timeline_flags: List[str] = field(default_factory=list)

## Recommended pipeline stages

1. **Ingest** claim files  
2. **Split** PDFs/images into pages  
3. **OCR** each page  
4. **Layout / document type classification**  
5. **Visual cue detection** (stamp/signature/QR/barcode/implant sticker/photo evidence)  
6. **Entity extraction** (dates, diagnosis, procedure, age, amounts, etc.)  
7. **Package-specific row creation**  
8. **Timeline construction**  
9. **Rules engine**  
10. **Explainable decisioning**

In [ ]:
# =========================
# 1. INGEST CLAIM FILES
# =========================

def iter_case_files(case_dir: Path) -> List[Path]:
    """Return all supported files inside one case folder, sorted by name."""
    case_dir = Path(case_dir)
    files: List[Path] = []
    for path in case_dir.rglob("*"):
        if not path.is_file():
            continue
        if path.name.startswith("."):
            continue
        if path.suffix.lower() in SUPPORTED_EXTENSIONS:
            files.append(path)
    return sorted(files, key=lambda p: p.name)


def discover_cases(data_root: Path) -> Dict[str, List[Path]]:
    """Walk Claims/<package_code>/<case_id>/<files>.

    Returns {case_id: [Path, ...]} and as a side effect populates the global
    PACKAGE_CODE_LOOKUP {case_id: package_code} so the cell-36 runner can
    resolve the package without any per-case helper file.
    """
    data_root = Path(data_root)
    cases: Dict[str, List[Path]] = {}

    if not data_root.exists():
        print(f"[discover_cases] DATA_ROOT does not exist: {data_root.resolve()}")
        return cases

    for pkg_dir in sorted(data_root.iterdir()):
        if not pkg_dir.is_dir():
            continue
        package_code = pkg_dir.name
        if package_code not in PACKAGE_CODES:
            continue
        for case_dir in sorted(pkg_dir.iterdir()):
            if not case_dir.is_dir():
                continue
            if case_dir.name.startswith("."):
                continue
            files = iter_case_files(case_dir)
            if not files:
                continue
            case_id = case_dir.name
            cases[case_id] = files
            PACKAGE_CODE_LOOKUP[case_id] = package_code

    return cases


In [ ]:
# =========================
# 2. SPLIT PDFS/IMAGES INTO PAGES
# =========================

def _resize_for_pipeline(image: Image.Image, max_side: int = MAX_IMAGE_SIDE) -> Image.Image:
    """Downscale so longest side <= max_side. Preserves aspect ratio."""
    if image.mode not in ("RGB", "L"):
        image = image.convert("RGB")
    w, h = image.size
    longest = max(w, h)
    if longest <= max_side:
        return image
    scale = max_side / float(longest)
    new_size = (max(1, int(w * scale)), max(1, int(h * scale)))
    return image.resize(new_size, Image.LANCZOS)


def extract_pages(file_path: Path) -> List[Dict[str, Any]]:
    """Rasterize a file into a list of page records.

    Returns [{"page_number": int, "image": PIL.Image, "file_name": str}].
    PDFs render at PDF_RENDER_ZOOM. Multi-frame TIFFs split into pages.
    Single images are wrapped as one page. All images are resized to
    MAX_IMAGE_SIDE on the longest dimension.
    """
    file_path = Path(file_path)
    ext = file_path.suffix.lower()
    file_name = file_path.name
    pages: List[Dict[str, Any]] = []

    if ext == ".pdf":
        with fitz.open(file_path) as doc:
            mat = fitz.Matrix(PDF_RENDER_ZOOM, PDF_RENDER_ZOOM)
            for i, page in enumerate(doc, start=1):
                pix = page.get_pixmap(matrix=mat, alpha=False)
                image = Image.open(io.BytesIO(pix.tobytes("png")))
                image.load()
                pages.append({
                    "page_number": i,
                    "image": _resize_for_pipeline(image),
                    "file_name": file_name,
                })
        return pages

    if ext in {".png", ".jpg", ".jpeg", ".bmp"}:
        with Image.open(file_path) as image:
            image.load()
            pages.append({
                "page_number": 1,
                "image": _resize_for_pipeline(image.copy()),
                "file_name": file_name,
            })
        return pages

    if ext in {".tif", ".tiff"}:
        with Image.open(file_path) as image:
            n_frames = getattr(image, "n_frames", 1)
            for i in range(n_frames):
                image.seek(i)
                pages.append({
                    "page_number": i + 1,
                    "image": _resize_for_pipeline(image.copy()),
                    "file_name": file_name,
                })
        return pages

    raise ValueError(f"Unsupported file type: {ext} ({file_path})")


In [ ]:
# =========================
# 3. OCR EACH PAGE
# =========================
# NOTE: cell-36 runner expects a dict ({"text": ..., "ocr_lines": [...]}) so
# this signature intentionally returns dict, not the tuple in the docstring.

def run_ocr(page_image: Any) -> Dict[str, Any]:
    """Run easyocr on a PIL image and return text + OCRLine list."""
    if page_image is None:
        return {"text": "", "ocr_lines": []}

    if not isinstance(page_image, Image.Image):
        try:
            page_image = Image.fromarray(np.array(page_image))
        except Exception:
            return {"text": "", "ocr_lines": []}

    img_np = np.array(page_image.convert("RGB"))

    try:
        raw = get_reader().readtext(img_np)
    except Exception as exc:
        print(f"[run_ocr] easyocr failed: {exc}")
        return {"text": "", "ocr_lines": []}

    lines: List[OCRLine] = []
    for bbox, text, confidence in raw:
        flat_bbox = [int(coord) for point in bbox for coord in point]
        lines.append(OCRLine(
            text=text,
            bbox=flat_bbox,
            confidence=round(float(confidence), 4),
        ))

    full_text = "\n".join(line.text for line in lines)
    return {"text": full_text, "ocr_lines": lines}


In [ ]:
# =========================
# 4. PAGE QUALITY
# =========================

def estimate_page_quality(page_image: Any, extracted_text: str) -> Dict[str, Any]:
    """Cheap cv2-based quality signals. Drives MG006A's poor_quality flag."""
    if page_image is None:
        return {
            "is_usable": False, "is_blurry": True, "is_low_contrast": True,
            "is_noisy": False, "is_low_text_density": True, "is_skewed": False,
            "blur_score": 0.0, "contrast_score": 0.0, "noise_score": 0.0,
            "text_density": 0.0, "skew_angle_deg": 0.0,
        }

    if not isinstance(page_image, Image.Image):
        page_image = Image.fromarray(np.array(page_image))

    img_np = np.array(page_image.convert("RGB"))
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    is_blurry = blur_score < 100.0

    contrast_score = float(gray.std())
    is_low_contrast = contrast_score < 30.0

    smoothed = cv2.GaussianBlur(gray, (5, 5), 0)
    noise_score = float(np.abs(gray.astype(np.float32) - smoothed).mean())
    is_noisy = noise_score > 10.0

    char_count = len((extracted_text or "").strip())
    pixel_area = max(1, h * w)
    text_density = char_count / pixel_area
    is_low_text_density = text_density < 1e-4

    skew_angle = 0.0
    try:
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=100)
        if lines is not None and len(lines) > 0:
            angles = [np.degrees(line[0][1]) - 90 for line in lines]
            skew_angle = float(np.median(angles))
    except Exception:
        skew_angle = 0.0
    is_skewed = abs(skew_angle) > 5.0

    is_usable = not (is_blurry and is_low_contrast and is_low_text_density)

    return {
        "is_usable": is_usable,
        "is_blurry": is_blurry,
        "is_low_contrast": is_low_contrast,
        "is_noisy": is_noisy,
        "is_low_text_density": is_low_text_density,
        "is_skewed": is_skewed,
        "blur_score": round(blur_score, 2),
        "contrast_score": round(contrast_score, 2),
        "noise_score": round(noise_score, 2),
        "text_density": round(text_density, 6),
        "skew_angle_deg": round(skew_angle, 2),
    }


In [ ]:
# =========================
# 5. VLM PAGE ANALYZER + VISUAL CUE DETECTION
# =========================
# The master VLM call lives here. It is invoked once per page (cached by
# image hash) and produces a structured JSON with doc_type, visual cues,
# evidence flags, dates, and age. detect_visual_elements (this cell) and
# classify_document_type (cell 15) both read from the cached response.

VISUAL_CUE_KEYS = [
    "has_stamp", "has_signature", "has_qr_code", "has_barcode",
    "has_implant_sticker", "has_photo_evidence", "has_handwriting",
    "has_table", "has_letterhead",
]

DATE_ROLES = ["pre_investigation", "post_investigation", "admission", "discharge", "other"]


def _image_to_data_url(image: Image.Image) -> Tuple[str, str]:
    """Returns (data_url, sha256_hex) for caching."""
    buf = io.BytesIO()
    image.convert("RGB").save(buf, format="JPEG", quality=85)
    raw = buf.getvalue()
    digest = hashlib.sha256(raw).hexdigest()
    b64 = base64.b64encode(raw).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}", digest


def _empty_vlm_response(package_code: str) -> Dict[str, Any]:
    return {
        "doc_type": "extra_document",
        "doc_type_confidence": 0.0,
        "corrected_text": "",
        "visual_cues": {k: False for k in VISUAL_CUE_KEYS},
        "evidence_flags": {k: {"value": 0, "quote": ""} for k in PACKAGE_EVIDENCE_KEYS.get(package_code, [])},
        "dates": [],
        "age_years": None,
        "is_extra_document": 1,
        "notes": "",
        "low_confidence": True,
    }


def _extract_json_object(text: str) -> Dict[str, Any]:
    """Robust JSON extractor: handles bare JSON, fenced ```json ... ```, and trailing prose."""
    if not isinstance(text, str):
        raise ValueError("VLM response not a string")
    text = text.strip()

    fenced = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    for cand in reversed(fenced):
        try:
            return json.loads(cand)
        except json.JSONDecodeError:
            continue

    decoder = json.JSONDecoder()
    starts = [i for i, ch in enumerate(text) if ch == "{"]
    for s in starts:
        try:
            obj, _ = decoder.raw_decode(text[s:])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue

    raise json.JSONDecodeError("no JSON object found in VLM response", text, 0)


def _build_vlm_prompt(package_code: str, easyocr_text: str) -> str:
    doc_types = PACKAGE_DOC_TYPES.get(package_code, [])
    evidence_keys = PACKAGE_EVIDENCE_KEYS.get(package_code, [])
    date_keys = PACKAGE_DATE_KEYS.get(package_code, [])

    evidence_block = ", ".join(evidence_keys) if evidence_keys else "(none)"
    date_block = ", ".join(date_keys) if date_keys else "(none expected)"
    visual_block = ",\n".join(f"    \"{k}\": true|false" for k in VISUAL_CUE_KEYS)

    disambiguators = {
        "MG064A": (
            "MG064A: distinguish clinical_notes (history/exam/condition), "
            "cbc_hb_report/post_hb_report (CBC or hemoglobin lab reports before/after treatment), "
            "treatment_details (transfusion or treatment record), indoor_case (admission/IPD record), "
            "and discharge_summary (final discharge document)."
        ),
        "SG039C": (
            "SG039C: distinguish operative_notes (procedure details, surgeon/anesthesia/operation findings) "
            "from discharge_summary (hospital course with admission/discharge and discharge advice); "
            "USG and LFT reports are investigation reports, not clinical notes."
        ),
        "MG006A": (
            "MG006A: distinguish clinical_notes (history/exam/symptoms) from vitals_treatment "
            "(temperature/vitals chart, medication, treatment monitoring); investigation_pre/post are "
            "lab/investigation reports before/after treatment, not generic notes."
        ),
        "SB039A": (
            "SB039A: distinguish xray_ct_knee (pre-op knee imaging/report), operative_notes "
            "(TKR surgery details), implant_invoice (implant bill/sticker), post_op_photo/post_op_xray "
            "(post-procedure visual evidence), and discharge_summary (final discharge document)."
        ),
    }
    package_hint = disambiguators.get(package_code, "")
    package_hint_block = f"Package disambiguation: {package_hint}\n\n" if package_hint else ""

    date_rules = {
        "MG006A": (
            "- For MG006A dates, emit pre_investigation only from investigation_pre pages and "
            "post_investigation only from investigation_post pages; do not extract these dates from "
            "clinical notes, vitals/treatment pages, or discharge summaries.\n"
        ),
        "SB039A": (
            "- For SB039A dates, emit admission and discharge dates only from discharge_summary pages; "
            "do not infer doa/dod from operative notes, invoices, imaging, or clinical notes.\n"
        ),
    }
    date_rule = date_rules.get(package_code, "")

    snippet = (easyocr_text or "").strip()
    if len(snippet) > 4000:
        snippet = snippet[:4000] + "\n...[truncated]"

    return (
        "You are an NHA medical-claim adjudicator. Analyze ONE page from a "
        f"claim under package {package_code}.\n\n"
        "An OCR engine returned this raw text (it may have errors or be "
        "incomplete; correct it using the image):\n"
        f"<ocr>\n{snippet}\n</ocr>\n\n"
        f"{package_hint_block}"
        "Work internally in this order: first identify doc_type, then is_extra_document, "
        "then evidence_flags with exact quotes, then dates. Do not output reasoning.\n\n"
        "Return ONLY a single JSON object (no prose, no markdown fences) with "
        "EXACTLY these keys:\n"
        "{\n"
        f"  \"doc_type\": one of [{', '.join(doc_types)}],\n"
        "  \"doc_type_confidence\": float in [0,1],\n"
        "  \"corrected_text\": full corrected text from the page,\n"
        f"  \"visual_cues\": {{\n{visual_block}\n  }},\n"
        f"  \"evidence_flags\": {{ for each of [{evidence_block}]: "
        "{\"value\": 0|1, \"quote\": \"<exact text from page or empty>\"} },\n"
        f"  \"dates\": [ each {{\"raw\": \"verbatim date substring from the page\", "
        f"\"role\": one of {DATE_ROLES} }} ],  // include only dates relevant to {date_block}\n"
        "  \"age_years\": integer or null,\n"
        "  \"is_extra_document\": 0|1,\n"
        "  \"notes\": short string (optional)\n"
        "}\n\n"
        "Rules:\n"
        f"- doc_type MUST be one of the listed labels for this package.\n"
        "- evidence_flags.value=1 only if you can quote supporting text.\n"
        "- evidence_flags.quote must be an exact substring from the page, not a paraphrase.\n"
        "- dates: extract only what is clearly written; raw must be the verbatim substring from the page.\n"
        f"{date_rule}"
        "- Set is_extra_document=1 if the page is a consent form, irrelevant lab, "
        "duplicate, or anything outside the STG-required documents for this package.\n"
        "- Only set is_extra_document=1 and doc_type=extra_document if the page clearly does "
        "not match any required document type for this package (e.g., blank page, consent form, "
        "payment receipt, unrelated document). When uncertain between two required doc types, "
        "pick the closest match rather than defaulting to extra_document.\n"
        "- Output ONLY the JSON object."
    )

def analyze_page_with_vlm(package_code: str, page_image: Image.Image, easyocr_text: str = "") -> Dict[str, Any]:
    """Single VLM call per page with on-disk cache keyed by image hash + package."""
    if page_image is None:
        return _empty_vlm_response(package_code)

    if not isinstance(page_image, Image.Image):
        page_image = Image.fromarray(np.array(page_image))

    data_url, img_hash = _image_to_data_url(page_image)
    cache_path = VLM_CACHE_DIR / f"{package_code}_{img_hash}.json"
    if cache_path.exists():
        try:
            return json.loads(cache_path.read_text(encoding="utf-8"))
        except Exception:
            pass  # corrupt cache -> re-run

    if nc is None:
        result = _empty_vlm_response(package_code)
        result["notes"] = "NHAclient not initialized; deterministic fallback."
        return result

    prompt = _build_vlm_prompt(package_code, easyocr_text)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": data_url}},
            {"type": "text", "text": prompt},
        ],
    }]

    parsed: Optional[Dict[str, Any]] = None
    last_err: Optional[str] = None
    for model in (VLM_MODEL, VLM_FALLBACK):
        try:
            response = nc.completion(model=model, messages=messages, metadata={"problem_statement": 1})
            content = response["choices"][0]["message"]["content"]
            parsed = _extract_json_object(content)
            parsed.setdefault("low_confidence", False)
            break
        except Exception as exc:
            last_err = f"{model}: {exc}"
            continue

    if parsed is None:
        result = _empty_vlm_response(package_code)
        result["notes"] = f"VLM hard-failed: {last_err}"
        try:
            cache_path.write_text(json.dumps(result), encoding="utf-8")
        except Exception:
            pass
        return result

    if parsed.get("doc_type") not in PACKAGE_DOC_TYPES.get(package_code, []):
        parsed["doc_type"] = "extra_document"
        parsed["is_extra_document"] = 1

    parsed.setdefault("visual_cues", {})
    for k in VISUAL_CUE_KEYS:
        parsed["visual_cues"].setdefault(k, False)

    parsed.setdefault("evidence_flags", {})
    for k in PACKAGE_EVIDENCE_KEYS.get(package_code, []):
        if k not in parsed["evidence_flags"]:
            parsed["evidence_flags"][k] = {"value": 0, "quote": ""}

    parsed.setdefault("dates", [])
    parsed.setdefault("age_years", None)
    parsed.setdefault("is_extra_document", int(parsed.get("doc_type") == "extra_document"))
    parsed.setdefault("corrected_text", "")
    parsed.setdefault("doc_type_confidence", 0.5)

    try:
        cache_path.write_text(json.dumps(parsed), encoding="utf-8")
    except Exception:
        pass
    return parsed


def detect_visual_elements(page_image: Any, extracted_text: str = "") -> Dict[str, Any]:
    """Return normalized visual cues.

    Internally calls analyze_page_with_vlm, caches the full response on the
    PageResult-shaped dict so cell 15 (classify_document_type) reuses it.
    """
    if page_image is None:
        return {k: False for k in VISUAL_CUE_KEYS}

    pkg = getattr(detect_visual_elements, "_current_package", None)
    if pkg is None:
        pkg = "MG064A"

    vlm = analyze_page_with_vlm(pkg, page_image, extracted_text)
    cues = dict(vlm.get("visual_cues", {}))
    for k in VISUAL_CUE_KEYS:
        cues.setdefault(k, False)
    cues["_vlm"] = vlm
    return cues


In [ ]:
# =========================
# 6. DOCUMENT CLASSIFICATION (reads cached VLM JSON)
# =========================
# Runner contract (cell 36): classify_document_type(package_code, text, visual_tags) -> dict
# We pull the VLM JSON that detect_visual_elements stashed in visual_tags["_vlm"];
# falls back to invoking the VLM directly if missing.

DOCUMENT_TYPES = [
    "clinical_notes",
    "cbc_hb_report",
    "indoor_case",
    "treatment_details",
    "post_hb_report",
    "discharge_summary",
    "usg_report",
    "lft_report",
    "operative_notes",
    "pre_anesthesia",
    "histopathology",
    "xray_ct_knee",
    "investigation_pre",
    "investigation_post",
    "vitals_treatment",
    "implant_invoice",
    "post_op_photo",
    "post_op_xray",
    "photo_evidence",
    "extra_document",
]


def classify_document_type(package_code: str, extracted_text: str, visual_tags: Dict[str, Any]) -> Dict[str, Any]:
    """Return classification + the full VLM evidence payload.

    The full payload is needed downstream by populate_row_for_package, so we
    return it under "vlm" rather than swallowing it.
    """
    vlm = (visual_tags or {}).get("_vlm")
    if not isinstance(vlm, dict):
        vlm = _empty_vlm_response(package_code)
        vlm["notes"] = "classify called without prior VLM call"

    doc_type = vlm.get("doc_type", "extra_document")
    if doc_type not in PACKAGE_DOC_TYPES.get(package_code, []):
        doc_type = "extra_document"

    confidence = float(vlm.get("doc_type_confidence", 0.0) or 0.0)
    return {
        "doc_type": doc_type,
        "confidence": confidence,
        "vlm": vlm,
    }


In [ ]:
# =========================
# ENTITY EXTRACTION HELPERS
# =========================

DATE_PATTERNS = [
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
    r"\b\d{1,2}-[A-Za-z]{3}-\d{2,4}\b",
    r"\b\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4}\b",
]


def find_dates(text: str) -> List[str]:
    """Return unique date-like substrings in order of appearance."""
    if not text:
        return []
    found: List[str] = []
    seen = set()
    for pattern in DATE_PATTERNS:
        for m in re.findall(pattern, text):
            if m not in seen:
                seen.add(m)
                found.append(m)
    return found


def find_age(text: str) -> Optional[int]:
    """Return parsed age in years, or None."""
    if not text:
        return None
    patterns = [
        r"\bage[:\s]+(\d{1,3})\b",
        r"\b(\d{1,3})\s*(?:years?|yrs?|y/o|yo)\b",
        r"\b(\d{1,3})\s*[/-]?\s*(?:M|F|Male|Female)\b",
    ]
    for pattern in patterns:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            try:
                age = int(m.group(1))
                if 0 < age < 120:
                    return age
            except (ValueError, IndexError):
                continue
    return None


def contains_any(text: str, keywords: List[str]) -> int:
    """1 if any keyword (case-insensitive substring) appears in text."""
    if not text or not keywords:
        return 0
    normalized = text.lower()
    return 1 if any(kw.lower() in normalized for kw in keywords) else 0


In [ ]:
# =========================
# 7. PAGE-TO-ROW MAPPING (canonical impl is in cell 19)
# =========================
# Intentionally empty - this duplicate stub in the original skeleton would
# shadow the real definition in cell 19 if filled in here. Cell 19 wins.

In [ ]:
# =========================
# PACKAGE ROW INITIALIZERS
# =========================

DATE_KEY_SET = {"pre_date", "post_date", "doa", "dod"}


def normalize_output_key(key: str) -> str:
    """Map internal aliases (s3_link / S3_link / S3_link/DocumentName) to 'link'."""
    return KEY_ALIASES.get(key, key)


def initialize_output_row(package_code: str, case_id: str, file_name: str, page_number: int) -> Dict[str, Any]:
    """Build an ordered row matching PACKAGE_SCHEMAS[package_code] exactly."""
    schema = PACKAGE_SCHEMAS[package_code]
    link_key = PACKAGE_LINK_KEY[package_code]
    row: Dict[str, Any] = {}
    for key in schema:
        if key == "case_id":
            row[key] = case_id
        elif key == link_key:
            row[key] = str(file_name)
        elif key == "procedure_code":
            row[key] = package_code
        elif key == "page_number":
            row[key] = int(page_number)
        elif key in DATE_KEY_SET:
            row[key] = None
        elif key == "document_rank":
            row[key] = 99
        else:
            row[key] = 0
    return row


In [ ]:
# =========================
# PAGE-TO-ROW MAPPING (canonical)
# =========================

def _select_date_for_role(dates: List[Dict[str, Any]], roles: List[str],
                          fallback_text: Optional[str] = None) -> Optional[str]:
    """Return the first VLM-extracted date whose role matches any in roles.

    Conservative regex backstop: when no role-matched date is found AND the
    VLM returned no dates at all AND fallback_text is provided, use the first
    pattern matched by find_dates(). The caller MUST only pass fallback_text
    when the page's doc_type strongly implies the role (e.g. an
    investigation_pre page being asked for pre_date) — otherwise the same
    fallback date could leak into multiple unrelated date slots.
    """
    for d in dates:
        if not isinstance(d, dict):
            continue
        if d.get("role") in roles:
            raw = d.get("raw")
            if raw:
                return str(raw)
    if fallback_text and not dates:
        regex_dates = find_dates(fallback_text)
        if regex_dates:
            return regex_dates[0]
    return None


def populate_row_for_package(package_code: str, page_result: PageResult) -> Dict[str, Any]:
    """Build the strict per-page row for one PageResult.

    Reads the cached VLM JSON from page_result.entities['vlm'] (set by
    process_case after classify_document_type runs). Falls back to safe
    defaults if missing.
    """
    case_id = page_result.case_id
    file_name = page_result.file_name
    page_number = page_result.page_number

    row = initialize_output_row(package_code, case_id, file_name, page_number)
    schema = PACKAGE_SCHEMAS[package_code]

    vlm = (page_result.entities or {}).get("vlm") or _empty_vlm_response(package_code)
    doc_type = page_result.doc_type or vlm.get("doc_type", "extra_document")
    if doc_type not in PACKAGE_DOC_TYPES.get(package_code, []):
        doc_type = "extra_document"

    # Document-type binary key
    if doc_type != "extra_document" and doc_type in row:
        row[doc_type] = 1

    # Package evidence flags from VLM
    evidence_flags = vlm.get("evidence_flags", {}) or {}
    for key in PACKAGE_EVIDENCE_KEYS.get(package_code, []):
        if key in row:
            v = evidence_flags.get(key, {})
            if isinstance(v, dict):
                row[key] = int(bool(v.get("value")))
            else:
                row[key] = int(bool(v))

    # Regex synonym backstop: if VLM said 0, but synonym hits, flip to 1.
    # Doc-type-gated keys (EVIDENCE_DOC_GATES) only fire on plausible doc types
    # so e.g. "calculi" mentioned in a clinical note doesn't flip usg_calculi.
    corrected_text = vlm.get("corrected_text") or page_result.extracted_text or ""
    syn_map = EVIDENCE_SYNONYMS.get(package_code, {})
    for key, synonyms in syn_map.items():
        if key not in row or row.get(key, 0) != 0 or not synonyms:
            continue
        gate = EVIDENCE_DOC_GATES.get(key)
        if gate is not None and doc_type not in gate:
            continue
        if contains_any(corrected_text, synonyms):
            row[key] = 1

    # SB039A: visual evidence of implant (sticker on report/photo) feeds
    # post_op_implant_present per ps.md ("report/image shows implant present").
    if package_code == "SB039A" and "post_op_implant_present" in row:
        visual_tags = page_result.visual_tags or {}
        if visual_tags.get("has_implant_sticker"):
            implant_doc_types = {"operative_notes", "implant_invoice", "post_op_xray",
                                 "post_op_photo", "discharge_summary"}
            if doc_type in implant_doc_types:
                row["post_op_implant_present"] = 1

    # Dates -> normalized DD-MM-YYYY into the package's date keys.
    # fallback_text is only passed when the page's doc_type strongly implies
    # the role we're extracting, so a regex-matched date can't leak into the
    # wrong slot.
    vlm_dates = vlm.get("dates", []) or []
    if package_code == "MG006A":
        if "pre_date" in row:
            fb = corrected_text if doc_type == "investigation_pre" else None
            row["pre_date"] = normalize_date(_select_date_for_role(vlm_dates, ["pre_investigation"], fb))
        if "post_date" in row:
            fb = corrected_text if doc_type == "investigation_post" else None
            row["post_date"] = normalize_date(_select_date_for_role(vlm_dates, ["post_investigation"], fb))
    elif package_code == "SB039A":
        # doa/dod come from discharge summary per ps.md; we can't disambiguate
        # the two via regex alone, so no fallback_text here.
        if "doa" in row:
            row["doa"] = normalize_date(_select_date_for_role(vlm_dates, ["admission"]))
        if "dod" in row:
            row["dod"] = normalize_date(_select_date_for_role(vlm_dates, ["discharge"]))

    # Age validity for SB039A
    if package_code == "SB039A" and "age_valid" in row:
        age = vlm.get("age_years")
        if age is None:
            age = find_age(corrected_text)
        if age is not None:
            lo, hi = PACKAGE_AGE_RANGE[package_code]
            row["age_valid"] = int(lo <= int(age) <= hi)

    # poor_quality for MG006A: VLM evidence OR cv2 quality
    if package_code == "MG006A" and "poor_quality" in row:
        from_vlm = row.get("poor_quality", 0)
        q = page_result.quality or {}
        from_quality = int(bool(q.get("is_blurry") or q.get("is_low_contrast") or q.get("is_low_text_density")))
        row["poor_quality"] = int(bool(from_vlm or from_quality))

    # extra_document and document_rank: doc_type is the source of truth.
    # The VLM's is_extra_document flag is only honored when doc_type itself
    # is "extra_document" (avoids stale empty-fallback flags overriding a
    # real classification).
    is_extra = int(doc_type == "extra_document")
    row["extra_document"] = is_extra
    row["document_rank"] = 99 if is_extra else int(infer_document_rank(package_code, row, doc_type) or 99)

    # Final order safety: re-emit in schema order in case any extra keys leaked in
    return {k: row.get(k) for k in schema}


In [ ]:
# =========================
# DOCUMENT RANKING
# =========================

RANK_MAP = {
    "MG064A": {
        "clinical_notes": 1,
        "cbc_hb_report": 2,
        "indoor_case": 2,
        "treatment_details": 3,
        "post_hb_report": 4,
        "discharge_summary": 5,
    },
    "SG039C": {
        "clinical_notes": 1,
        "usg_report": 2,
        "lft_report": 3,
        "pre_anesthesia": 4,
        "operative_notes": 5,
        "discharge_summary": 5,
        "histopathology": 6,
        "photo_evidence": 6,
    },
    "MG006A": {
        "clinical_notes": 1,
        "investigation_pre": 2,
        "vitals_treatment": 3,
        "investigation_post": 4,
        "discharge_summary": 5,
    },
    "SB039A": {
        "clinical_notes": 1,
        "xray_ct_knee": 2,
        "indoor_case": 3,
        "operative_notes": 4,
        "implant_invoice": 5,
        "post_op_photo": 6,
        "post_op_xray": 6,
        "discharge_summary": 7,
    }
}

def infer_document_rank(package_code: str, row: Dict[str, Any], doc_type: str) -> int:
    """Return per-page rank from RANK_MAP. Defaults to 99 (extra)."""
    if doc_type == "extra_document":
        return 99
    return int(RANK_MAP.get(package_code, {}).get(doc_type, 99))


def propagate_ranks_within_documents(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Multi-page document rule: every page of one source file gets the same
    document_rank (the modal non-99 rank assigned to its pages). Extra-doc
    pages keep 99.
    """
    if not rows:
        return rows

    link_keys = {"link", "S3_link", "S3_link/DocumentName", "s3_link"}
    grouped: Dict[str, List[int]] = defaultdict(list)
    for idx, row in enumerate(rows):
        file_name = next((row[k] for k in link_keys if k in row), None)
        case_id = row.get("case_id", "")
        grouped[f"{case_id}::{file_name}"].append(idx)

    for _, idxs in grouped.items():
        if len(idxs) <= 1:
            continue
        ranks = [rows[i].get("document_rank") for i in idxs if rows[i].get("document_rank") not in (None, 99)]
        if not ranks:
            continue
        modal = Counter(ranks).most_common(1)[0][0]
        for i in idxs:
            if rows[i].get("extra_document") == 1:
                continue
            rows[i]["document_rank"] = int(modal)
    return rows


In [ ]:
# =========================
# 8. TIMELINE CONSTRUCTION
# =========================

def _parse_normalized_date(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.strptime(s, "%d-%m-%Y")
    except (ValueError, TypeError):
        return None


def build_episode_timeline(package_code: str, page_results: List[PageResult]) -> List[TimelineEvent]:
    """Order events by (rank, parsed_date). One event per page-with-evidence."""
    events: List[Dict[str, Any]] = []
    for pr in page_results:
        row = pr.output_row or {}
        if row.get("extra_document") == 1:
            continue
        rank = row.get("document_rank", 99)
        if rank == 99:
            continue

        vlm = (pr.entities or {}).get("vlm") or {}
        dates = vlm.get("dates", []) or []
        date_str: Optional[str] = None
        for d in dates:
            if isinstance(d, dict) and d.get("raw"):
                date_str = normalize_date(str(d["raw"]))
                if date_str:
                    break

        events.append({
            "rank": int(rank),
            "parsed": _parse_normalized_date(date_str),
            "date": date_str,
            "doc_type": pr.doc_type,
            "source": f"{pr.file_name}#p{pr.page_number}",
            "evidence": vlm.get("notes", ""),
        })

    events.sort(key=lambda e: (e["rank"], e["parsed"] or datetime.max))

    procedure_dt: Optional[datetime] = None
    for e in events:
        if e["doc_type"] in {"operative_notes", "treatment_details", "vitals_treatment"} and e["parsed"]:
            procedure_dt = e["parsed"]
            break

    timeline: List[TimelineEvent] = []
    for i, e in enumerate(events, start=1):
        validity = "Valid"
        if e["parsed"] and procedure_dt:
            if e["rank"] <= 2 and e["parsed"] > procedure_dt:
                validity = "Out of order (pre-procedure event after procedure)"
            elif e["rank"] >= 4 and e["parsed"] < procedure_dt:
                validity = "Out of order (post-procedure event before procedure)"
        timeline.append(TimelineEvent(
            sequence=i,
            event_type=e["doc_type"],
            date=e["date"],
            source_document=e["source"],
            temporal_validity=validity,
            evidence={"rank": e["rank"], "notes": e["evidence"]},
        ))
    return timeline


In [ ]:
# =========================
# 9. RULES ENGINE (STARTER)
# =========================

MANDATORY_DOCS = {
    "MG064A": ["clinical_notes", "cbc_hb_report", "indoor_case", "treatment_details", "post_hb_report", "discharge_summary"],
    "SG039C": ["clinical_notes", "usg_report", "lft_report", "operative_notes", "pre_anesthesia", "discharge_summary", "photo_evidence", "histopathology"],
    "MG006A": ["clinical_notes", "investigation_pre", "vitals_treatment", "investigation_post", "discharge_summary"],
    "SB039A": ["clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes", "implant_invoice", "post_op_photo", "post_op_xray", "discharge_summary"],
}

def aggregate_case_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Collapse per-page binary flags via OR; collect non-null dates.

    Date selection is role-aware (per ps.md): doa/dod prefer the row whose
    doc_type is discharge_summary; pre_date/post_date prefer the matching
    investigation row. Falls back to the first non-null date across all
    rows when no preferred source row is available.
    """
    if not rows:
        return {}
    skip_keys = {"case_id", "link", "S3_link", "S3_link/DocumentName", "s3_link",
                 "procedure_code", "page_number", "document_rank"}
    date_keys = {"pre_date", "post_date", "doa", "dod"}
    date_source_pref = {
        "doa": "discharge_summary",
        "dod": "discharge_summary",
        "pre_date": "investigation_pre",
        "post_date": "investigation_post",
    }

    case_evidence: Dict[str, Any] = {}

    for row in rows:
        for k, v in row.items():
            if k in skip_keys or k in date_keys:
                continue
            if isinstance(v, (int, float)) and v == 1:
                case_evidence[k] = 1
            else:
                case_evidence.setdefault(k, 0)

    for dk in date_keys:
        chosen: Optional[str] = None
        pref_doc = date_source_pref.get(dk)
        if pref_doc:
            for row in rows:
                if row.get(pref_doc) == 1 and row.get(dk):
                    chosen = row[dk]
                    break
        if chosen is None:
            for row in rows:
                if row.get(dk):
                    chosen = row[dk]
                    break
        case_evidence[dk] = chosen

    return case_evidence


def run_rules_engine(case_id: str, package_code: str,
                     rows: List[Dict[str, Any]],
                     timeline: List[TimelineEvent]) -> ClaimDecision:
    """Per-package STG checks + missing-doc check -> PASS/CONDITIONAL/FAIL."""
    evidence = aggregate_case_rows(rows)
    reasons: List[str] = []
    rule_flags: List[str] = []
    timeline_flags: List[str] = []

    mandatory = MANDATORY_DOCS.get(package_code, [])
    missing = [doc for doc in mandatory if not evidence.get(doc)]

    hard_fail = False
    soft_fail = False

    if package_code == "MG064A":
        if not evidence.get("severe_anemia"):
            rule_flags.append("severe_anemia not documented")
            soft_fail = True
        if not (evidence.get("common_signs") or evidence.get("significant_signs") or evidence.get("life_threatening_signs")):
            rule_flags.append("no anemia signs documented")
            soft_fail = True

    elif package_code == "SG039C":
        if not evidence.get("clinical_condition"):
            rule_flags.append("no qualifying STG clinical condition")
            soft_fail = True
        if not evidence.get("usg_calculi"):
            rule_flags.append("USG evidence of calculi missing")
            soft_fail = True
        if not evidence.get("pain_present"):
            rule_flags.append("pain not documented")
            soft_fail = True

    elif package_code == "MG006A":
        if not evidence.get("fever"):
            rule_flags.append("fever not documented")
            soft_fail = True
        if not evidence.get("symptoms"):
            rule_flags.append("no qualifying STG symptoms documented")
            soft_fail = True
        if evidence.get("post_date") and evidence.get("pre_date"):
            pre_dt = _parse_normalized_date(evidence["pre_date"])
            post_dt = _parse_normalized_date(evidence["post_date"])
            if pre_dt and post_dt and post_dt < pre_dt:
                timeline_flags.append("post_date before pre_date")
                hard_fail = True

    elif package_code == "SB039A":
        if not evidence.get("arthritis_type"):
            rule_flags.append("arthritis_type not documented")
            soft_fail = True
        if not evidence.get("post_op_implant_present"):
            rule_flags.append("post-op implant evidence missing")
            soft_fail = True
        if not evidence.get("age_valid"):
            rule_flags.append("age outside STG window for TKR")
            soft_fail = True
        if evidence.get("doa") and evidence.get("dod"):
            doa_dt = _parse_normalized_date(evidence["doa"])
            dod_dt = _parse_normalized_date(evidence["dod"])
            if doa_dt and dod_dt and dod_dt < doa_dt:
                timeline_flags.append("dod before doa")
                hard_fail = True

    for ev in timeline or []:
        if ev.temporal_validity and ev.temporal_validity != "Valid":
            timeline_flags.append(f"{ev.event_type}: {ev.temporal_validity}")

    if missing:
        reasons.append(f"missing mandatory docs: {missing}")
    reasons.extend(rule_flags)
    reasons.extend(timeline_flags)

    n_mandatory = max(1, len(mandatory))
    completeness = 1.0 - (len(missing) / n_mandatory)

    if hard_fail or len(missing) > n_mandatory // 2:
        decision = DECISION_FAIL
        confidence = round(0.4 * completeness, 2)
    elif missing or soft_fail or timeline_flags:
        decision = DECISION_CONDITIONAL
        confidence = round(0.6 * completeness, 2)
    else:
        decision = DECISION_PASS
        confidence = round(0.9 * completeness, 2)

    return ClaimDecision(
        case_id=case_id,
        package_code=package_code,
        decision=decision,
        confidence=confidence,
        reasons=reasons or ["all checks passed"],
        missing_documents=missing,
        rule_flags=rule_flags,
        timeline_flags=timeline_flags,
    )


In [ ]:
# =========================
# 10. EXPLAINABLE DECISIONING
# =========================

def build_human_readable_summary(package_code: str, page_results: List[PageResult],
                                 decision: ClaimDecision) -> pd.DataFrame:
    """One row per page: file, page, doc_type, confidence, key evidence, notes."""
    rows = []
    for pr in page_results:
        row_data = pr.output_row or {}
        evidence_quotes: List[str] = []
        vlm = (pr.entities or {}).get("vlm") or {}
        for k, v in (vlm.get("evidence_flags") or {}).items():
            if isinstance(v, dict) and v.get("value") and v.get("quote"):
                evidence_quotes.append(f"{k}: \"{v['quote']}\"")

        notes = "; ".join(decision.reasons or []) if decision and pr.page_number == 1 else ""

        rows.append({
            "case_id": pr.case_id,
            "file": pr.file_name,
            "page": pr.page_number,
            "doc_type": pr.doc_type,
            "doc_type_confidence": round(pr.doc_type_confidence or 0.0, 3),
            "extra_document": row_data.get("extra_document", 0),
            "document_rank": row_data.get("document_rank", 99),
            "evidence": " | ".join(evidence_quotes)[:500],
            "notes": notes,
        })
    return pd.DataFrame(rows)


def build_timeline_df(timeline: List[TimelineEvent]) -> pd.DataFrame:
    """Convert TimelineEvent list to DataFrame matching the ps.md sample table."""
    if not timeline:
        return pd.DataFrame(columns=["sequence", "event_type", "date", "source_document", "temporal_validity"])
    return pd.DataFrame([{
        "sequence": e.sequence,
        "event_type": e.event_type,
        "date": e.date,
        "source_document": e.source_document,
        "temporal_validity": e.temporal_validity,
    } for e in timeline])


In [ ]:
# =========================
# CORE PIPELINE DRIVER
# =========================


def _as_pil_rgb(page_image: Any) -> Optional[Image.Image]:
    if page_image is None:
        return None
    if isinstance(page_image, Image.Image):
        return page_image.convert("RGB")
    try:
        return Image.fromarray(np.array(page_image)).convert("RGB")
    except Exception:
        return None


def _preprocess_page_for_ocr_vlm(page_image: Any, quality: Dict[str, Any]) -> Any:
    """Apply only quality-triggered cleanup before OCR and VLM.

    The original page remains the source for final quality flags; this image is
    just a clearer view for text extraction and multimodal classification.
    """
    pil_img = _as_pil_rgb(page_image)
    if pil_img is None:
        return page_image

    img_np = np.array(pil_img)
    changed = False

    skew_angle = float((quality or {}).get("skew_angle_deg") or 0.0)
    if np.isfinite(skew_angle) and 10.0 < abs(skew_angle) < 45.0:
        h, w = img_np.shape[:2]
        center = (w / 2.0, h / 2.0)
        matrix = cv2.getRotationMatrix2D(center, -skew_angle, 1.0)
        img_np = cv2.warpAffine(
            img_np,
            matrix,
            (w, h),
            flags=cv2.INTER_CUBIC,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(255, 255, 255),
        )
        changed = True

    if float((quality or {}).get("contrast_score", 100.0)) < 20.0:
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced_l = clahe.apply(l_channel)
        img_np = cv2.cvtColor(cv2.merge((enhanced_l, a_channel, b_channel)), cv2.COLOR_LAB2RGB)
        changed = True

    return Image.fromarray(img_np) if changed else pil_img


def _process_one_page(package_code: str, case_id: str, page: Dict[str, Any]) -> PageResult:
    """OCR -> quality -> visual cues (VLM) -> classify (cached VLM) -> row."""
    page_image = page.get("image")
    file_name = page.get("file_name", "")
    page_number = int(page.get("page_number", 1))

    pr = PageResult(case_id=case_id, file_name=file_name, page_number=page_number)

    pre_quality = estimate_page_quality(page_image, "") or {}
    ocr_vlm_image = _preprocess_page_for_ocr_vlm(page_image, pre_quality)

    ocr_out = run_ocr(ocr_vlm_image)
    pr.extracted_text = ocr_out.get("text", "") or ""
    pr.ocr_lines = ocr_out.get("ocr_lines", []) or []

    pr.quality = estimate_page_quality(page_image, pr.extracted_text) or {}
    original_skew = float(pre_quality.get("skew_angle_deg") or 0.0)
    pr.quality["preprocess_applied"] = bool(
        (np.isfinite(original_skew) and 10.0 < abs(original_skew) < 45.0)
        or float(pre_quality.get("contrast_score", 100.0)) < 20.0
    )

    detect_visual_elements._current_package = package_code
    pr.visual_tags = detect_visual_elements(ocr_vlm_image, pr.extracted_text) or {}

    cls = classify_document_type(package_code, pr.extracted_text, pr.visual_tags) or {}
    pr.doc_type = cls.get("doc_type", "extra_document")
    pr.doc_type_confidence = float(cls.get("confidence", 0.0) or 0.0)
    pr.entities = {"vlm": cls.get("vlm", {})}

    pr.output_row = populate_row_for_package(package_code, pr)
    return pr


def process_case(case_id: str, files: List[Path], package_code: str) -> Dict[str, Any]:
    """Full pipeline for one case. Pages are processed concurrently."""
    page_inputs: List[Tuple[Path, Dict[str, Any]]] = []
    for fp in files:
        try:
            pages = extract_pages(fp) or []
        except Exception as exc:
            print(f"  [extract_pages failed] {fp.name}: {exc}")
            continue
        for p in pages:
            page_inputs.append((fp, p))

    page_results: List[Optional[PageResult]] = [None] * len(page_inputs)

    with ThreadPoolExecutor(max_workers=VLM_CONCURRENCY) as ex:
        future_to_idx = {
            ex.submit(_process_one_page, package_code, case_id, page): idx
            for idx, (_fp, page) in enumerate(page_inputs)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                page_results[idx] = future.result()
            except Exception as exc:
                _fp, page = page_inputs[idx]
                print(f"  [page failed] {page.get('file_name')}#{page.get('page_number')}: {exc}")
                fallback = PageResult(
                    case_id=case_id,
                    file_name=page.get("file_name", _fp.name),
                    page_number=int(page.get("page_number", 1)),
                )
                fallback.entities = {"vlm": _empty_vlm_response(package_code)}
                fallback.output_row = populate_row_for_package(package_code, fallback)
                page_results[idx] = fallback

    page_results = [pr for pr in page_results if pr is not None]
    page_results.sort(key=lambda pr: (pr.file_name, pr.page_number))

    strict_rows = [pr.output_row for pr in page_results]
    strict_rows = propagate_ranks_within_documents(strict_rows)
    for pr, row in zip(page_results, strict_rows):
        pr.output_row = row

    timeline = build_episode_timeline(package_code, page_results)
    decision = run_rules_engine(case_id, package_code, strict_rows, timeline)
    summary_df = build_human_readable_summary(package_code, page_results, decision)
    timeline_df = build_timeline_df(timeline)

    return {
        "case_id": case_id,
        "package_code": package_code,
        "page_results": page_results,
        "strict_rows": strict_rows,
        "timeline": timeline,
        "decision": decision,
        "summary_df": summary_df,
        "timeline_df": timeline_df,
    }


In [ ]:
# =========================
# BATCH RUNNER (thin wrapper; the canonical entry point is cell 36)
# =========================

def run_batch(data_root: Path, package_code_lookup: Optional[Dict[str, str]] = None) -> Dict[str, Any]:
    """Iterate cases, run process_case, return {case_id: case_result}."""
    cases = discover_cases(data_root)
    lookup = package_code_lookup or PACKAGE_CODE_LOOKUP
    results: Dict[str, Any] = {}
    for i, (case_id, files) in enumerate(cases.items(), start=1):
        package_code = lookup.get(case_id)
        if package_code not in PACKAGE_CODES:
            continue
        try:
            results[case_id] = process_case(case_id, files, package_code)
            print(f"[{i}/{len(cases)}] {package_code}/{case_id} ok ({len(results[case_id]['strict_rows'])} pages)")
        except Exception as exc:
            print(f"[{i}/{len(cases)}] {package_code}/{case_id} FAILED: {exc}")
    return results


In [ ]:
# =========================
# DEMO WITH THE PROVIDED EXAMPLE JSON STRUCTURES
# =========================

EXAMPLE_JSON_PATHS = {
    "SG039C": "data/SG039C_Cholecystectomy.json",
    "SB039A": "data/SB039A_Knee_Replacement.json",
    "MG064A": "data/MG064A_Anemia.json",
    "MG006A": "data/MG006A_Fever.json",
}

def load_example_jsons() -> Dict[str, List[Dict[str, Any]]]:
    out = {}
    for pkg, path in EXAMPLE_JSON_PATHS.items():
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                out[pkg] = json.load(f)
    return out

example_jsons = load_example_jsons()
{k: len(v) for k, v in example_jsons.items()}

In [ ]:
# =========================
# EXPORTERS
# =========================

def _safe_case_filename(case_id: str) -> str:
    """Make a case_id safe for filesystem use."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", case_id)


def export_case_outputs(case_result: Dict[str, Any], output_root: Path = OUTPUT_ROOT) -> None:
    """Write strict JSON rows + summary CSV + timeline CSV + decision JSON."""
    case_id = case_result["case_id"]
    safe = _safe_case_filename(case_id)

    strict_dir = output_root / "strict"
    summary_dir = output_root / "summary"
    timeline_dir = output_root / "timeline"
    decision_dir = output_root / "decisions"
    for d in (strict_dir, summary_dir, timeline_dir, decision_dir):
        d.mkdir(parents=True, exist_ok=True)

    (strict_dir / f"{safe}.json").write_text(
        json.dumps(case_result["strict_rows"], indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

    summary_df: pd.DataFrame = case_result.get("summary_df")
    if summary_df is not None and not summary_df.empty:
        summary_df.to_csv(summary_dir / f"{safe}.csv", index=False)

    timeline_df: pd.DataFrame = case_result.get("timeline_df")
    if timeline_df is not None and not timeline_df.empty:
        timeline_df.to_csv(timeline_dir / f"{safe}.csv", index=False)

    decision = case_result["decision"]
    decision_payload = asdict(decision) if hasattr(decision, "__dataclass_fields__") else dict(decision)
    (decision_dir / f"{safe}.json").write_text(
        json.dumps(decision_payload, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )


In [ ]:
# =========================
# VALIDATOR FOR EXACT OUTPUT KEYS
# =========================

def validate_output_rows(package_code: str, rows: List[Dict[str, Any]]) -> Tuple[bool, List[str]]:
    expected = PACKAGE_SCHEMAS[package_code]
    issues = []

    for i, row in enumerate(rows):
        row_keys = list(row.keys())
        if row_keys != expected:
            issues.append(
                f"Row {i}: key order / names mismatch.\nExpected: {expected}\nGot:      {row_keys}"
            )
    return len(issues) == 0, issues

In [ ]:
# =========================
# EXAMPLE: VALIDATE ORGANIZER JSON SAMPLES
# =========================

for pkg, rows in example_jsons.items():
    ok, issues = validate_output_rows(pkg, rows)
    print(pkg, "->", "VALID" if ok else "INVALID")
    if issues:
        print("\n".join(issues[:2]))

## Suggested participant extensions

Participants can improve this scaffold by adding:

- multilingual OCR
- VLM-based page classification
- stamp/signature/implant sticker detection
- table extraction for invoices / vitals
- robust date normalization to `DD-MM-YYYY`
- exact STG rule encoding per package
- evidence spans and bounding boxes
- calibrated confidence scoring
- duplicate / redundant / extra document tagging
- page grouping into document-level clusters

In [ ]:
# =========================
# DATE NORMALIZATION UTIL
# =========================

from datetime import datetime

def normalize_date(date_str: Optional[str]) -> Optional[str]:
    if not date_str:
        return None

    candidates = [
        "%d/%m/%y", "%d/%m/%Y",
        "%d-%m-%y", "%d-%m-%Y",
        "%d-%b-%y", "%d-%b-%Y",
        "%d %b %Y", "%d %B %Y",
        "%m/%d/%y", "%m/%d/%Y",  # keep only if your data needs it
    ]

    cleaned = str(date_str).strip()
    for fmt in candidates:
        try:
            dt = datetime.strptime(cleaned, fmt)
            return dt.strftime("%d-%m-%Y")
        except Exception:
            continue
    return None

In [ ]:
# =========================
# OPTIONAL: POST-PROCESS DATES INTO REQUIRED FORMAT
# =========================

def normalize_dates_in_rows(package_code: str, rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    date_keys = []
    if package_code == "MG006A":
        date_keys = ["pre_date", "post_date"]
    elif package_code == "SB039A":
        date_keys = ["doa", "dod"]

    normalized = []
    for row in rows:
        r = dict(row)
        for dk in date_keys:
            if dk in r:
                r[dk] = normalize_date(r.get(dk))
        normalized.append(r)
    return normalized

In [ ]:
# Intentionally empty - the working normalize_date is in cell 31.
# The skeleton's stub here would shadow cell 31 if executed top-to-bottom.

In [ ]:
# Intentionally empty - the working normalize_dates_in_rows is in cell 32.

In [ ]:
# =========================
# SAMPLE DECISION REPORT RENDERER
# =========================

def render_decision_report(case_result: Dict[str, Any]) -> None:
    """Print a compact reviewer card for one case."""
    decision = case_result.get("decision")
    if decision is None:
        print("(no decision available)")
        return

    print("=" * 60)
    print(f"Case:       {decision.case_id}")
    print(f"Package:    {decision.package_code}")
    print(f"Decision:   {decision.decision}  (confidence={decision.confidence})")
    if decision.missing_documents:
        print(f"Missing:    {', '.join(decision.missing_documents)}")
    if decision.rule_flags:
        print(f"Rule flags: {'; '.join(decision.rule_flags)}")
    if decision.timeline_flags:
        print(f"Timeline:   {'; '.join(decision.timeline_flags)}")
    print(f"Reasons:    {'; '.join(decision.reasons)}")
    print("=" * 60)

In [ ]:

# =========================
# MAIN RUNNER / FINAL ASSEMBLY
# =========================
# Note: discover_cases (cell 10) populates PACKAGE_CODE_LOOKUP as a side
# effect, so we don't need to seed it here. Pages within each case are
# parallelized inside process_case via ThreadPoolExecutor(VLM_CONCURRENCY).

import time as _time

cases = discover_cases(DATA_ROOT)

if not cases:
    print(f"No cases found under: {DATA_ROOT.resolve()}")
    print("Add data first, then rerun this section.")
else:
    BATCH_RESULTS = {}
    FINAL_STRICT_OUTPUTS = {}
    FINAL_PAGE_RESULTS = {}
    FINAL_DECISIONS = {}
    FINAL_SUMMARIES = {}
    FINAL_TIMELINES = {}

    total = len(cases)
    for idx, (case_id, files) in enumerate(cases.items(), start=1):
        package_code = PACKAGE_CODE_LOOKUP.get(case_id)
        if package_code not in PACKAGE_CODES:
            print(f"[{idx}/{total}] skipping '{case_id}' (no valid package code)")
            continue

        t0 = _time.time()
        try:
            case_result = process_case(case_id, files, package_code)
        except Exception as exc:
            print(f"[{idx}/{total}] {package_code}/{case_id} HARD FAIL: {exc}")
            traceback.print_exc()
            continue

        BATCH_RESULTS[case_id] = case_result
        FINAL_STRICT_OUTPUTS[case_id] = case_result["strict_rows"]
        FINAL_PAGE_RESULTS[case_id] = [asdict(pr) for pr in case_result["page_results"]]
        FINAL_DECISIONS[case_id] = asdict(case_result["decision"])
        FINAL_SUMMARIES[case_id] = case_result["summary_df"]
        FINAL_TIMELINES[case_id] = case_result["timeline_df"]

        try:
            export_case_outputs(case_result, OUTPUT_ROOT)
        except Exception as exc:
            print(f"  export warning: {exc}")

        ok, issues = validate_output_rows(package_code, case_result["strict_rows"])
        elapsed = _time.time() - t0
        status = "OK" if ok else "INVALID"
        print(f"[{idx}/{total}] {package_code}/{case_id}  "
              f"{len(files)} files / {len(case_result['strict_rows'])} pages / "
              f"{elapsed:.1f}s  schema={status}  decision={case_result['decision'].decision}")
        if not ok:
            for issue in issues[:2]:
                print(f"    {issue}")

    final_path = OUTPUT_ROOT / "FINAL_STRICT_OUTPUTS.json"
    final_path.write_text(
        json.dumps(FINAL_STRICT_OUTPUTS, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )
    print(f"\nFinished {len(BATCH_RESULTS)} case(s). Wrote {final_path}")

# =========================
# FINAL RESULTS DISPLAY
# =========================

if "BATCH_RESULTS" in globals() and BATCH_RESULTS:
    print("\nAvailable final objects:")
    print("- BATCH_RESULTS")
    print("- FINAL_STRICT_OUTPUTS")
    print("- FINAL_PAGE_RESULTS")
    print("- FINAL_DECISIONS")
    print("- FINAL_SUMMARIES")
    print("- FINAL_TIMELINES")

    all_rows = []
    for case_id, rows in FINAL_STRICT_OUTPUTS.items():
        for row in rows:
            all_rows.append(row)

    FINAL_STRICT_OUTPUTS_DF = pd.DataFrame(all_rows)
    print("\nCombined strict output preview:")
    display(FINAL_STRICT_OUTPUTS_DF.head())

    decision_rows = []
    for case_id, decision_dict in FINAL_DECISIONS.items():
        decision_rows.append(decision_dict)

    FINAL_DECISIONS_DF = pd.DataFrame(decision_rows)
    print("\nDecision preview:")
    display(FINAL_DECISIONS_DF.head())
else:
    print("No final results available yet.")